In [ ]:
#Loading both manifests and combine:
import pandas as pd
import shutil
import os
from pathlib import Path
from PIL import Image
from tqdm import tqdm

chex_manifest = pd.read_csv("../data/subsets/chexpert_subset_manifest.csv")
isic_manifest = pd.read_csv("../data/subsets/isic_subset_manifest.csv")

combined = pd.concat([chex_manifest, isic_manifest]).reset_index(drop=True)
print(combined.shape)
print(combined["dataset"].value_counts())

(18776, 6)
dataset
chexpert    8776
Name: count, dtype: int64


In [ ]:
#createing destination folder
DEST_ROOT = Path("../data/subsets/images")
(DEST_ROOT / "chexpert").mkdir(parents=True, exist_ok=True)
(DEST_ROOT / "isic").mkdir(parents=True, exist_ok=True)

In [ ]:
#Copy all files, verifying readability, and tracking failures:
CHEXPERT_RAW = Path("../data/raw/chexpert")
ISIC_RAW = Path("../data/raw/isic")

new_paths = []
failed = []

for idx, row in tqdm(combined.iterrows(), total=len(combined)):
    if row["dataset"] == "chexpert":
        src = CHEXPERT_RAW / row["image_path"]
        dst = DEST_ROOT / "chexpert" / f"{row['image_id']}"
    else:
        src = ISIC_RAW / row["image_path"]
        dst = DEST_ROOT / "isic" / f"{row['image_id']}.jpg"

    try:
        img = Image.open(src)
        img.verify()  # this confirms file isn't corrupted
        shutil.copy(src, dst)
        new_paths.append(str(dst))
    except Exception as e:
        failed.append((row["image_id"], str(e)))
        new_paths.append(None)

combined["final_path"] = new_paths
print(f"Failed: {len(failed)}")
if failed:
    print(failed[:10])

  0%|          | 0/18776 [00:00<?, ?it/s]

100%|██████████| 18776/18776 [00:35<00:00, 526.06it/s]

Failed: 0
